In [ ]:
using Pkg 
Pkg.activate("..")

In [ ]:
include("../src/PhasorNetworks.jl")
using .PhasorNetworks, Plots, DifferentialEquations

In [ ]:
using Lux, MLUtils, OneHotArrays, Statistics, Test, LuxCUDA
using Random: Xoshiro, AbstractRNG
using Base: @kwdef
using Zygote: withgradient
using Optimisers, ComponentArrays
using Statistics: mean
using LinearAlgebra: diag
using PhasorNetworks: bind
using Distributions: Normal
using DifferentialEquations: Heun, Tsit5

In [ ]:
solver_args = Dict(:adaptive => false, :dt => 0.01)

spk_args = SpikingArgs(threshold = 0.001,
                    solver=Tsit5(), 
                    solver_args = solver_args)

In [ ]:
cdev = cpu_device()
gdev = gpu_device()

In [ ]:
#global args for all tests
n_x = 101
n_y = 101
n_vsa = 1
epsilon = 0.10
repeats = 10
epsilon = 0.025


tspan = (0.0, repeats*1.0)
tbase = collect(tspan[1]:spk_args.solver_args[:dt]:tspan[2])

@kwdef mutable struct Args
    η::Float64 = 3e-4       ## learning rate
    batchsize::Int = 256    ## batch size
    epochs::Int = 10        ## number of epochs
    use_cuda::Bool = false   ## use gpu (if cuda available)
    rng::Xoshiro = Xoshiro(42) ## global rng
end

In [ ]:
c_args = Args(use_cuda = false)
g_args = Args(use_cuda = true)

# Static

In [ ]:
x_in = random_symbols((64, 64, 1, 4), c_args.rng)

In [ ]:
layer = Chain(PhasorConv((32, 32), 1 => 1, complex_to_angle, init_bias=default_bias),)

In [ ]:
sol_layer = Chain(PhasorConv((32, 32), 1 => 1, complex_to_angle, return_solution=true),)

In [ ]:
ps, st = Lux.setup(c_args.rng, layer)

In [ ]:
function zero_weights(nt::NamedTuple)
    return NamedTuple{keys(nt)}(map(zero_weights, values(nt)))
end

function zero_weights(x::AbstractArray)
    return x
end

function zero_weights(x)
    return x
end

function zero_weights(nt::NamedTuple{(:bias_real,)})
    return (bias_real = zeros(size(nt.bias_real)),)
end

function zero_weights(nt::NamedTuple{(:bias_imag,)})
    return (bias_imag = zeros(size(nt.bias_imag)),)
end

function zero_weights(nt::NamedTuple{names}) where names
    new_nt = nt
    if :bias_real in names
        new_nt = merge(new_nt, (bias_real = zeros(size(nt.bias_real)),))
    end
    if :bias_imag in names
        new_nt = merge(new_nt, (bias_imag = zeros(size(nt.bias_imag)),))
    end
    if new_nt === nt
        return NamedTuple{names}(map(zero_weights, values(nt)))
    else
        return new_nt
    end
end


In [ ]:
ps_nb = zero_weights(ps)

In [ ]:
y, _ = layer(x_in, ps, st)

In [ ]:
yu, _ = layer(x_in, ps_nb, st)

In [ ]:
histogram(vec(y))

In [ ]:
histogram(vec(yu))

In [ ]:
ps_g = gdev(ps)
st_g = gdev(st)
ps_gu = gdev(ps_nb)
x_in_g = gdev(x_in);

In [ ]:
y_g, _ = layer(x_in_g, ps_g, st_g)

In [ ]:
y_gu, _ = layer(x_in_g, ps_gu, st_g)

In [ ]:
histogram(vec(ps.layer_1.layer.weight))

In [ ]:
y2 = y_g |> cdev
y2u = y_gu |> cdev

In [ ]:
histogram(vec(y2))

In [ ]:
# check cpu-gpu deviation

In [ ]:
maximum(abs.(y .- y2))

In [ ]:
maximum(abs.(yu .- y2u))

# Dynamic

In [ ]:
st_x = phase_to_train(x_in, spk_args=spk_args, repeats=10)

In [ ]:
st_xg = SpikeTrainGPU(st_x)

In [ ]:
y_t, _ = layer(SpikingCall(st_x, spk_args, (0.0, 10.0)), ps, st)

In [ ]:
y_sol, _ = sol_layer(SpikingCall(st_x, spk_args, (0.0, 10.0)), ps, st)

In [ ]:
y_tu, _ = layer(SpikingCall(st_x, spk_args, (0.0, 10.0)), ps_nb, st);

In [ ]:
u = y_sol[1] |> stack;

In [ ]:
size(u)

In [ ]:
uref = phase_to_potential.(0.0f0, y_sol[2], offset=0.0, spk_args=spk_args);

In [ ]:
size(uref)

In [ ]:
size(u)

In [ ]:
u_p = potential_to_phase(u, tbase, spk_args=spk_args) |> stack;

In [ ]:
size(u_p)

In [ ]:
plot(real.(u[1,1,1,1,:]), imag.(u[1,1,1,1,:]))

In [ ]:
plot(tbase, u_p[1,1,1,1,:])

In [ ]:
plot(sort(y_t.train.times))

In [ ]:
plot(sort(y_tu.train.times))

In [ ]:
y_tp = train_to_phase(y_t);

In [ ]:
y_tpu = train_to_phase(y_tu);

In [ ]:
spk_args.spk_scale

In [ ]:
histogram(vec(y_tpu[5,:,:,:,:] .- yu))

In [ ]:
y_tg, _ = layer(SpikingCall(st_xg, spk_args, (0.0, 10.0)), ps_g, st_g)

In [ ]:
y_tgu, _ = layer(SpikingCall(st_xg, spk_args, (0.0, 10.0)), ps_gu, st);

In [ ]:
y_tpg = train_to_phase(y_tg);

In [ ]:
y_tpgu = train_to_phase(y_tgu);

In [ ]:
plot(sort(y_tgu.train.times |> cdev))

In [ ]:
histogram(vec(y_tpgu[5,:,:,:,:] .- yu))

In [ ]:
scatter(vec(y_tp[4,:,:,:,:]), vec(y_tpg[4,:,:,:,:]))

In [ ]:
histogram(vec(y .- y_tpg[8,:,:,:,:]))

In [ ]:
histogram(y_tpg[8,:,:,:,:] |> vec)

In [ ]:
scatter(vec(y_tp[8,:,:,:,:]), vec(y_tpg[8,:,:,:,:]))

In [ ]:
size(y)

In [ ]:
size(y_tp)

In [ ]:
size(y_tpg)

In [ ]:
plot(cycle_correlation(reshape(y, 33^2,4), reshape(y_tp, (10,33^2,4))), label = "CPU")
plot!(cycle_correlation(reshape(y, 33^2,4), reshape(y_tpg, (11,33^2,4))), label="GPU")